# 17 — Stronger baselines for win/loss (boosting & calibration)

**Goal:** On the **same event-based train/test split** as notebooks 13/15, compare **Random Forest** (your existing default) to:

- **HistGradientBoostingClassifier** — sklearn’s native histogram boosting; often stronger than RF on tabular data with interactions, fewer hyperparameters than XGBoost.
- **(Optional) XGBoost** — industry standard gradient boosted trees; `pip install xgboost` if the import cell succeeds.
- **CalibratedClassifierCV** — wraps a base estimator with **isotonic** or **sigmoid** calibration to improve **Brier score** and **log loss** when raw scores are miscalibrated.
- **Logistic regression (L2)** — linear baseline on the same Z-delta features; strong when the signal is roughly additive.

**Why these methods?** Random forests average deep trees and can be **under-confident or lumpy** in probability space. Boosting often improves **ranking (AUC)**; **calibration** explicitly adjusts predicted probabilities toward observed frequencies.

**Inputs:** `ufc_fight_stats_cleaned.csv`, `ufc_gmm_comparison.csv`  
**Target:** `Win_A` (first corner wins) on fights where both fighters have Z-scores.

**Note:** For **multiclass fight dynamics** (six method buckets), see notebook **16**; you can swap in XGBoost there using the same feature matrix pattern.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

fights = pd.read_csv("../data/processed/ufc_fight_stats_cleaned.csv")
styles = pd.read_csv("../data/processed/ufc_gmm_comparison.csv")
z_cols = ["Sig_Str_PM_Z", "Takedown_Att_PM_Z", "Sub_Att_PM_Z", "Control_Ratio_Z"]
z_map = styles[["Fighter"] + z_cols].set_index("Fighter")

f1 = fights.drop_duplicates("Fight_Id", keep="first").copy()
f1 = f1.merge(z_map, left_on="Fighter", right_index=True, how="inner")
f1 = f1.merge(
    z_map, left_on="Opponent", right_index=True, how="inner", suffixes=("", "_B")
)
f1["Fight_Id"] = f1["Fight_Id"].astype(str)
f1["Win_A"] = f1["Won"].astype(int)
ufz = f1.rename(columns={"Fighter": "Fighter_A", "Opponent": "Fighter_B"}).copy()
for c in z_cols:
    ufz[f"delta_{c}"] = ufz[c] - ufz[f"{c}_B"]
feat = [f"delta_{c}" for c in z_cols]

sorted_events = np.sort(ufz["Event_Id_x"].unique())
split_i = max(1, int(0.8 * len(sorted_events)))
train_ev = set(sorted_events[:split_i])
tr = ufz["Event_Id_x"].isin(train_ev)
te = ~tr

Xtr, Xte = ufz.loc[tr, feat], ufz.loc[te, feat]
ytr, yte = ufz.loc[tr, "Win_A"], ufz.loc[te, "Win_A"]
print(f"Train {len(ytr)}  Test {len(yte)}")


Train 4444  Test 1098


In [2]:
EPS = 1e-6


def eval_model(name, clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    p = np.clip(p, EPS, 1 - EPS)
    return {
        "model": name,
        "auc": roc_auc_score(yte, p),
        "brier": brier_score_loss(yte, p),
        "logloss": log_loss(yte, p),
        "acc": accuracy_score(yte, (p >= 0.5).astype(int)),
    }


rows = []
rows.append(
    eval_model(
        "RandomForest (13-style)",
        RandomForestClassifier(
            n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
        ),
        Xtr,
        ytr,
        Xte,
        yte,
    )
)
rows.append(
    eval_model(
        "HistGradientBoosting",
        HistGradientBoostingClassifier(
            max_iter=250, max_depth=8, learning_rate=0.06, random_state=42
        ),
        Xtr,
        ytr,
        Xte,
        yte,
    )
)
rows.append(
    eval_model(
        "LogisticRegression (L2, lbfgs)",
        LogisticRegression(
            max_iter=1000, C=1.0, random_state=42, solver="lbfgs"
        ),
        Xtr,
        ytr,
        Xte,
        yte,
    )
)

# Calibration: refit base HGB on train folds inside CV — use subset for speed
base_hgb = HistGradientBoostingClassifier(
    max_iter=200, max_depth=8, learning_rate=0.06, random_state=42
)
cal = CalibratedClassifierCV(base_hgb, method="isotonic", cv=3)
rows.append(eval_model("HGB + isotonic calibration", cal, Xtr, ytr, Xte, yte))

try:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
    )
    rows.append(eval_model("XGBoost", xgb, Xtr, ytr, Xte, yte))
except ImportError:
    print("XGBoost not installed — skip. Install with: pip install xgboost")

res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


                         model    auc  brier  logloss    acc
       RandomForest (13-style) 0.6039 0.2347   0.6611 0.6075
          HistGradientBoosting 0.5522 0.2567   0.7152 0.5619
LogisticRegression (L2, lbfgs) 0.6240 0.2315   0.6545 0.6120
    HGB + isotonic calibration 0.5799 0.2372   0.6666 0.5947
                       XGBoost 0.5783 0.2454   0.6852 0.5747


### Reading the table

- **AUC** — discrimination (ranking); boosting often gains a few points here.
- **Brier / log loss** — probability quality; **calibration** often helps even when AUC is flat.
- **Logistic regression** — if it is close to boosting, your signal is roughly **linear** in Z-deltas; large gaps suggest **nonlinear** interactions (where trees help).

For **dynamics** (notebook **16**), the same estimators apply; six-way classification is **harder**—consider **ordinal** or **hierarchical** models (first predict finish vs. decision, then method) as a thesis extension.